### [Stock Arbiter outperforms a Stock Screener](https://medium.com/illumination/dec03e7cc21d)

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

# arbitrary mock order book of new strategy mentions, use your actual order book later
rows = [
  {"asset":"AAPL", "strategy":"mr_1", "dir":"long", "w":1.0},
  {"asset":"AAPL", "strategy":"brk_2", "dir":"long", "w":1.0},
  {"asset":"AAPL", "strategy":"trend_1", "dir":"short", "w":1.0},
  {"asset":"MSFT", "strategy":"mr_1", "dir":"long", "w":1.0},
  {"asset":"MSFT", "strategy":"brk_2", "dir":"long", "w":1.0},
  {"asset":"TSLA", "strategy":"trend_1", "dir":"short", "w":1.2},
]

df = pd.DataFrame(rows)
display(df)

,asset,strategy,dir,w
0,AAPL,mr_1,long,1.0
1,AAPL,brk_2,long,1.0
2,AAPL,trend_1,short,1.0
3,MSFT,mr_1,long,1.0
4,MSFT,brk_2,long,1.0
5,TSLA,trend_1,short,1.2


In [2]:
all_strats = sorted(df["strategy"].unique())
S = max(1, len(all_strats))

In [3]:
def arbiter(g):
    wl = g.loc[g.dir.eq("long"), "w"].sum()
    ws = g.loc[g.dir.eq("short"), "w"].sum()
    tot = max(1e-9, wl + ws)
    vote = (wl - ws) / tot                         # [-1, 1]
    contr = min(wl, ws) / tot                      # [0, 0.5] typically
    div = g["strategy"].nunique() / S              # [0, 1]
    div_conf = 0.5 + 0.5 * div                     # [0.5, 1.0]
    raw = vote * div_conf - 0.4 * contr            # keep it simple
    score = float(np.tanh(1.6 * raw))              # squash to [-1, 1]
    conf = int(round(abs(score) * 100))
    action = "NO_TRADE"
    if conf >= 60:
        action = "LONG" if score > 0 else "SHORT"
    return pd.Series({"score": score, "conf": conf, "vote": vote, "contr": contr, "div": div, "action": action})

In [4]:
out = df.groupby("asset", sort=False).apply(arbiter).reset_index()
display(out.sort_values(["conf", "score"], ascending=False)) ## to_string(index=False)

,asset,score,conf,vote,contr,div,action
1,MSFT,0.870062,87,1.000000,0.000000,0.666667,LONG
2,TSLA,-0.788202,79,-1.000000,0.000000,0.333333,SHORT
0,AAPL,0.309507,31,0.333333,0.333333,1.000000,NO_TRADE


### [Stock Prediction Model](https://wire.insiderfinance.io/building-a-stock-prediction-model-using-python-0334eba05d44)

In [5]:
import yfinance as yf

data = yf.download("AAPL", start="2020-01-01", end="2024-01-01", progress=False, auto_adjust=True)
display(data.sample(20))

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2021-03-17,121.515663,122.587057,119.158589,120.824127,111932600
2021-07-09,141.576553,142.103398,139.176448,139.274019,99890800
2023-06-26,182.876907,185.620997,182.837416,184.416755,48088700
2021-12-27,176.459732,176.547797,173.269704,173.289264,74919600
2023-10-10,176.323563,177.638158,175.888657,176.036929,43698000
2020-03-16,58.524212,62.600436,57.990217,58.461387,322423600
2023-09-08,176.115967,178.152117,175.730485,176.284011,65551300
2023-06-09,178.622604,179.876189,178.296865,179.155623,48900000


In [6]:
data['MA50'] = data['Close'].rolling(window=50).mean()
data['MA200'] = data['Close'].rolling(window=200).mean()
data['Returns'] = data['Close'].pct_change()

display(data.sample(20))

Price,Close,High,Low,Open,Volume,MA50,MA200,Returns
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,,,
Date,,,,,,,,
2021-04-22,128.508972,130.661494,127.992756,129.580358,84566500,123.076421,117.182652,-0.011686
2021-02-05,133.203629,133.846469,132.327038,133.778297,75693800,125.892770,105.647814,-0.003098
2023-01-13,132.633072,132.790550,129.582008,129.946164,57809700,137.310275,147.116064,0.010119
2023-02-24,144.614944,145.088086,143.639077,145.009226,55469600,138.390168,144.677258,-0.018005
2022-12-29,127.564377,128.420641,125.714051,125.969942,75703700,141.034202,149.014571,0.028324
2020-04-29,69.523033,69.991788,68.595190,68.798155,137280800,65.639320,NaN,0.032845
2022-05-05,153.600815,160.763037,151.817600,160.537691,130525300,162.085758,156.144909,-0.055716
2022-03-01,159.900818,163.232094,158.695687,161.370495,83474400,167.357798,148.782884,-0.011628


In [7]:
data['Target'] = (data['Close'].shift(-1) > data['Close']).astype(int)
display(data.sample(20))

Price,Close,High,Low,Open,Volume,MA50,MA200,Returns,Target
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,,,,
Date,,,,,,,,,
2021-03-26,118.058006,118.320989,115.827555,117.220369,94071200,125.152820,113.624722,0.005141,1
2020-04-30,70.989685,71.166075,69.672828,70.061843,183064000,65.495214,NaN,0.021096,0
2020-08-19,112.336632,113.749246,112.241976,112.603622,145538000,94.885134,NaN,0.001255,1
2021-04-22,128.508972,130.661494,127.992756,129.580358,84566500,123.076421,117.182652,-0.011686,1
2020-12-29,131.166809,134.979174,130.651363,134.259502,121047300,117.014772,96.890726,-0.013315,0
2023-04-06,162.308609,162.604328,159.686591,160.110443,45390100,151.160011,146.676689,0.005496,0
2021-04-29,130.008926,133.505582,129.005712,132.921179,151101000,123.087521,118.124216,-0.000748,0
2020-04-08,64.289406,64.603517,63.119937,63.484787,168895200,68.407029,NaN,0.025595,1


In [8]:
from sklearn.model_selection import train_test_split

features = ['MA50', 'MA200', 'Returns']
X = data[features].dropna()
y = data['Target'].loc[X.index]

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False)

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print(f"Score: {model.score(X_test, y_test)}")

Score: 0.4752475247524752


In [10]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions)}")

Accuracy: 0.4752475247524752


In [11]:
data.loc[X_test.index, 'Prediction'] = predictions
data['Strategy_Returns'] = data['Returns'] * data['Prediction']

display(data.sample(20))

Price,Close,High,Low,Open,Volume,MA50,MA200,Returns,Target,Prediction,Strategy_Returns
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,,,,,,
Date,,,,,,,,,,,
2020-04-13,66.024277,66.133011,64.231409,64.830644,131022800,67.897510,NaN,0.019628,1,NaN,NaN
2021-02-26,118.106697,121.603337,118.048252,119.402105,164560400,128.410342,109.537368,0.002231,1,NaN,NaN
2021-05-25,123.809990,125.195419,123.244111,124.707586,72009500,124.513779,120.530667,-0.001573,0,NaN,NaN
2022-12-23,129.778854,130.330013,127.593891,128.853688,63814900,141.785297,149.372267,-0.002798,0,NaN,NaN
2021-06-10,123.039223,125.068577,122.873364,123.927060,71186400,125.366539,121.058746,-0.008023,1,NaN,NaN
2023-03-20,155.152283,155.566298,151.948694,152.855569,73641400,144.433337,145.237412,0.015484,1,1.0,0.015484
2020-12-03,119.564346,120.381278,118.854387,120.128415,78967600,113.311774,92.169279,-0.001138,0,NaN,NaN
2023-06-29,187.141098,187.614909,186.499500,186.637691,46347300,173.726027,152.051796,0.001797,1,0.0,0.000000
